# День 1 · usage, temperature, токены RU/EN, галлюцинация

Четыре коротких эксперимента с одним и тем же клиентом: как выглядит `usage` в ответе, как
`temperature` меняет разброс, сколько токенов стоит одна и та же мысль на русском и английском,
и как модель ведёт себя, когда не знает ответа, — вместо честного «не знаю» она может уверенно
сочинить правдоподобную неправду. Это и есть галлюцинация.

In [ ]:
import labkit  # noqa: F401  читает .env
from client import make_client, MODEL

# --- НАСТРОЙКИ ---
QUESTION = "Придумай название для сервиса, который отвечает на вопросы по документам компании."
TEMPERATURES = (0.0, 1.0)     # 0 — почти детерминированно, 1 — разнообразно
REPEATS = 3                   # сколько раз задать один вопрос при каждой температуре
SAMPLES = (                   # пара фраз одинакового смысла для сравнения токенов
    "The quick brown fox jumps over the lazy dog near the river.",
    "Быстрая рыжая лиса перепрыгивает через ленивую собаку у реки.",
)

client = make_client()

## Один запрос, обёрнутый в функцию

`ask()` — не абстракция ради абстракции: она нужна, чтобы вызвать один и тот же запрос много раз
подряд ниже (для разных `temperature` и с разным вопросом) без копирования кода вызова.

In [ ]:
def ask(prompt: str, temperature: float):
    """Один запрос к модели. Возвращает текст ответа и usage — счётчики токенов."""
    r = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        max_tokens=120,                                   # потолок длины ответа
        messages=[
            {"role": "system", "content": "Отвечай одним коротким предложением, без пояснений."},  # правила
            {"role": "user", "content": prompt},                                                    # вопрос
        ],
    )
    if not r.choices or not r.choices[0].message.content or r.usage is None:
        raise RuntimeError("нет текста/usage: проверь отказ, лимит ответа и возможности провайдера")
    return r.choices[0].message.content.strip(), r.usage

## Temperature: одна и та же модель, разный разброс

`temperature` масштабирует распределение вероятностей следующего токена перед выбором. При `0.0`
модель почти всегда берёт самый вероятный токен — три запроса одного и того же вопроса дают
одинаковый или почти одинаковый ответ. При `1.0` распределение более плоское, и три ответа обычно
заметно расходятся. Смотри на разброс формулировок в выводе ниже, а не на «правильность»: тут нет
правильного ответа, есть только разброс.

In [ ]:
print(f"модель: {MODEL}")
for t in TEMPERATURES:
    print(f"\n=== temperature={t} ===")
    for i in range(REPEATS):
        text, usage = ask(QUESTION, t)
        print(f"{i + 1}. {text}   [in={usage.prompt_tokens} out={usage.completion_tokens}]")   # in — токены запроса, out — ответа

## Токены: русский против английского

Модель режет текст на токены алгоритмом вроде BPE, а не по словам. Ниже — одна и та же мысль на
двух языках, `max_tokens=1` — ответ не нужен, нужен только `prompt_tokens`, то есть честный счётчик
входа. Замечено на практике: русский текст обычно занимает больше токенов, чем английский такого же
смысла, — это прямо влияет на цену и на то, сколько текста поместится в контекстное окно (ниже,
в `04_cost.py`, ты увидишь этот бюджет в действии).

In [ ]:
print("\n=== токены: русский против английского ===")
for s in SAMPLES:
    r = client.chat.completions.create(model=MODEL, max_tokens=1, messages=[{"role": "user", "content": s}])  # ответ не нужен, нужен подсчёт входа
    print(f"{r.usage.prompt_tokens:4d} токенов | {len(s):3d} символов | {s}")

## Галлюцинация: модель не знает, что не знает

LLM предсказывает наиболее вероятный следующий токен — она не сверяется с реальной базой фактов и
не отличает «я знаю» от «я не знаю» так, как это делает человек. Если спросить про правдоподобно
звучащий, но несуществующий параметр хорошо известной библиотеки, модель нередко не откажется, а
уверенно опишет поведение и даже значение по умолчанию — потому что «описание параметра известной
функции» очень сильный паттерн в обучающих данных, и достроить его правдоподобно проще, чем
распознать, что конкретно этого параметра не существует. У `pydantic.BaseModel.model_validate()`
(её ты вызывал в шаге 4) нет параметра `dedupe` — реальная сигнатура: `strict`, `from_attributes`,
`context`. Спросим про несуществующий параметр напрямую.

In [ ]:
print("\n=== галлюцинация: спрашиваем про несуществующий параметр pydantic ===")
r = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    temperature=0,
    messages=[{"role": "user", "content": "Что делает параметр dedupe в pydantic.BaseModel.model_validate() и какое у него значение по умолчанию?"}],
)
print(r.choices[0].message.content)
print("\nУ model_validate() в pydantic v2 нет параметра dedupe: только strict, from_attributes, context.")

Если модель уверенно описала поведение `dedupe` и назвала конкретное значение по умолчанию — это и
есть галлюцинация: она не сверилась с реальной сигнатурой pydantic, а достроила правдоподобный ответ
по шаблону «как обычно описывают параметры валидации». Если она честно ответила, что такого
параметра нет, — запиши это в `results.md` и попробуй другой несуществующий параметр: у разных
моделей и разных вопросов вероятность галлюцинации разная, но она не нулевая почти никогда. Лечится
не запретом, а конструкцией: RAG подмешивает реальные факты в контекст вместо догадки (день 2),
честный отказ явно разрешён в промпте, а для кода — проверка сигнатуры реальным `inspect` или тестом,
не доверие тексту ответа.